# <font color="#418FDE" size="6.5" uppercase>**Daten erkunden**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Erstellen aussagekräftige Tabellenübersichten und Verteilungskennzahlen. 
- Visualisieren Verteilungen, Zusammenhänge und Gruppenunterschiede. 
- Erkennen Ausreißer, fehlende Werte und Korrelation-ohne-Kausalität als EDA-Risiken. 


## **1. Tabellen schnell verstehen**

### **1.1. Erster Tabellenüberblick**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_A/image_01_01.jpg?v=1787630918" width="250">



>* Zeilen, Spalten und Bedeutung zuerst klären
>* Struktur fachlich prüfen, typische Fehler vermeiden

>* Spalten, Datentypen und Beispielwerte prüfen
>* Tabellenform passend zur Analyse einschätzen

>* Vollständigkeit, Plausibilität und Konsistenz früh prüfen
>* Datenstruktur mit Neugier und Skepsis hinterfragen



In [ ]:
#@title Python-Code - Erster Tabellenüberblick

# Wir erstellen einen ersten kompakten Tabellenüberblick.
# Der Fokus liegt auf Struktur und Datentypen.
# Am Ende sehen wir zentrale Prüfhinweise.

import pandas as pd

# Kleine Beispieltabelle mit typischen EDA-Stolperstellen.
orders = pd.DataFrame(
    {
        "order_id": [101, 102, 103, 104, 105],
        "customer_group": ["Neu", "Stamm", "Neu", "Stamm", "Neu"],
        "order_value_eur": [49.9, 120.0, 0.0, 89.5, 250.0],
        "delivery_days": [3, 5, 0, 12, 4],
        "returned": ["Nein", "Ja", "Nein", "nein", "Nein"],
    }
)

# Diese Prüfung schützt vor einer leeren Tabelle.
if orders.shape[0] == 0 or orders.shape[1] == 0:
    raise ValueError("Die Beispieltabelle darf nicht leer sein.")

# Ein erster Überblick beginnt mit Größe und Spaltennamen.
print(f"Form: {orders.shape[0]} Zeilen, {orders.shape[1]} Spalten")
print("Spalten: " + ", ".join(orders.columns))

# Datentypen zeigen, wie pandas die Spalten interpretiert.
dtype_text = orders.dtypes.astype(str).to_dict()
print("Datentypen: " + str(dtype_text))

# Wenige Beispielzeilen reichen für eine erste Orientierung.
print("Erste drei Zeilen:")
print(orders.head(3).to_string(index=False))

# Ein kurzer Qualitätscheck markiert frühe Analysefragen.
missing_total = int(orders.isna().sum().sum())
zero_values = int((orders["order_value_eur"] == 0).sum())
returned_variants = orders["returned"].nunique()

# Die Zusammenfassung verbindet Struktur, Typen und Plausibilität.
print(f"Fehlende Werte gesamt: {missing_total}")
print(f"Bestellungen mit Warenwert 0 Euro: {zero_values}")
print(f"Schreibvarianten in returned: {returned_variants}")



### **1.2. Kennzahlen gezielt vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_A/image_01_02.jpg?v=1787630916" width="250">



>* Kennzahlen fassen Tabelleneigenschaften gezielt zusammen
>* Median ergänzt Mittelwert bei Ausreißern

>* Kennzahlen immer im Kontext deuten
>* Streuung und Extreme zeigen Datenprobleme

>* Kennzahlen nach sinnvollen Gruppen vergleichen
>* Zahlen kritisch als Fragenanstoß nutzen



In [ ]:
#@title Python-Code - Kennzahlen gezielt vergleichen

# Wir vergleichen Kennzahlen für Lieferzeiten gezielt.
# Mittelwert und Median zeigen unterschiedliche Perspektiven.
# Die Grafik macht Ausreißer sofort sichtbar.

import pandas as pd
import matplotlib.pyplot as plt

# Diese kleine Tabelle enthält normale Werte und Ausreißer.
delivery_days = [2, 3, 3, 4, 4, 5, 5, 6, 7, 21]
orders = pd.DataFrame({"delivery_days": delivery_days})

# Eine einfache Prüfung verhindert leere Auswertungen.
if len(orders) == 0:
    raise ValueError("Die Tabelle enthält keine Zeilen.")

# Diese Kennzahlen fassen dieselbe Spalte unterschiedlich zusammen.
summary = orders["delivery_days"].agg(["mean", "median", "min", "max", "std"])
summary["range"] = summary["max"] - summary["min"]

# Quartile zeigen, wo die mittleren fünfzig Prozent liegen.
quartiles = orders["delivery_days"].quantile([0.25, 0.75])
summary["q1"] = quartiles.loc[0.25]
summary["q3"] = quartiles.loc[0.75]

# Die Ausgabe bleibt kurz und auf die Interpretation fokussiert.
print("Kennzahlen für Lieferzeiten in Tagen:")
print(summary.round(1).to_string())
print("Merke: Der Ausreißer erhöht den Mittelwert stärker als den Median.")

# Ein Boxplot verbindet Kennzahlen mit der sichtbaren Verteilung.
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(orders["delivery_days"], vert=False, showmeans=True)

# Achsenbeschriftungen machen die Einheit eindeutig.
ax.set_title("Lieferzeiten: Median, Streuung und Ausreißer")
ax.set_xlabel("Tage bis zur Lieferung")
ax.set_yticks([1])
ax.set_yticklabels(["Bestellungen"])

# Diese Hilfslinien markieren zwei zentrale Kennzahlen.
ax.axvline(summary["mean"], color="orange", linestyle="--", label="Mittelwert")
ax.axvline(summary["median"], color="green", linestyle=":", label="Median")
ax.legend()

plt.show()



### **1.3. Kategorien zählen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_A/image_01_03.jpg?v=1787630920" width="250">



>* Häufigkeiten zeigen wichtige Kategorien im Datensatz
>* Sie prüfen Struktur und Vergleichbarkeit

>* Absolute Zahlen immer mit Anteilen vergleichen
>* Kategorien auf Schreibweisen und Leerwerte prüfen

>* Fragestellungen machen Kategorienzählungen aussagekräftiger
>* Häufigkeit zeigt keine Ursache oder Wichtigkeit



In [ ]:
#@title Python-Code - Kategorien zählen

# Wir zählen Kategorien in einer kleinen Tabelle.
# Absolute und relative Häufigkeiten werden gemeinsam gezeigt.
# Ein Balkendiagramm macht dominante Gruppen sofort sichtbar.

import pandas as pd
import matplotlib.pyplot as plt

# Diese Beispieldaten enthalten typische Kundensegmente.
data = pd.DataFrame(
    {
        "customer_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
        "segment": ["Privat", "Firma", "Privat", "Student", "Privat", "Firma", "Privat", "Unbekannt", "Student", "Privat"],
    }
)

# Wir prüfen zuerst die erwartete Tabellengröße.
if len(data) == 0:
    raise ValueError("Die Tabelle enthält keine Zeilen.")

# value_counts zählt jede Kategorie zuverlässig.
counts = data["segment"].value_counts(dropna=False)
shares = data["segment"].value_counts(normalize=True, dropna=False) * 100

# Eine kompakte Übersicht verbindet Anzahl und Anteil.
summary = pd.DataFrame(
    {"Anzahl": counts, "Anteil in Prozent": shares.round(1)}
)

print("Häufigkeitsübersicht nach Kundensegment:")
print(summary.to_string())
print("Größte Gruppe: " + summary.index[0])

# Das Diagramm zeigt dieselben Zählungen visuell.
fig, ax = plt.subplots(figsize=(7, 4))
summary["Anzahl"].plot(kind="bar", ax=ax, color="steelblue")

ax.set_title("Kategorien zählen: Kundensegmente")
ax.set_xlabel("Kundensegment")
ax.set_ylabel("Anzahl der Zeilen")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()



## **2. Diagramme gezielt nutzen**

### **2.1. Verteilungen sichtbar machen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_A/image_02_01.jpg?v=1787630922" width="250">



>* Diagramme zeigen Muster, Streuung und Auffälligkeiten
>* Visualisierungen regen wichtige EDA-Fragen an

>* Numerische Daten: Histogramm, Dichteplot oder Boxplot
>* Kategorien: Balken passend zur Analysefrage

>* Diagramme können Muster zeigen oder verzerren
>* Kontext prüfen und weiter analysieren



In [ ]:
#@title Python-Code - Verteilungen sichtbar machen

# Dieses Beispiel macht Verteilungen mit einem Histogramm sichtbar.
# Es zeigt Häufungen, Streuung und ungewöhnliche Werte.
# Am Ende entsteht ein beschriftetes Diagramm.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen kleine, nachvollziehbare Beispieldaten.
rng = np.random.default_rng(42)
regular_orders = rng.gamma(shape=2.2, scale=18.0, size=180)
large_orders = rng.normal(loc=145.0, scale=18.0, size=12)

# Die Werte werden zu einer Spalte zusammengeführt.
order_values = np.concatenate([regular_orders, large_orders])
order_values = np.clip(order_values, 5.0, None)

# Eine kurze Prüfung schützt vor leeren Daten.
if order_values.size == 0:
    raise ValueError("Die Beispieldaten enthalten keine Werte.")

# Eine Tabelle hilft beim Vergleich mit dem Diagramm.
summary = pd.Series(order_values).describe()
median_value = float(np.median(order_values))
mean_value = float(np.mean(order_values))

print("Beispiel: tägliche Bestellwerte in Euro")
print(f"Anzahl Beobachtungen: {order_values.size}")
print(f"Mittelwert: {mean_value:.1f} Euro")
print(f"Median: {median_value:.1f} Euro")
print(f"Maximum: {summary['max']:.1f} Euro")

# Das Histogramm zeigt, wo viele Werte liegen.
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(order_values, bins=18, color="#4C78A8", edgecolor="white")
ax.axvline(median_value, color="#F58518", linewidth=2, label="Median")

# Beschriftungen machen die Aussage des Diagramms klar.
ax.set_title("Verteilung synthetischer Bestellwerte")
ax.set_xlabel("Bestellwert in Euro")
ax.set_ylabel("Anzahl Beobachtungen")
ax.legend()

plt.show()



### **2.2. Zusammenhänge und Trends**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_A/image_02_02.jpg?v=1787630923" width="250">



>* Streudiagramme zeigen gemeinsame Veränderungen von Merkmalen
>* Muster geben Hinweise für weitere Analysen

>* Richtung, Stärke und Form getrennt betrachten
>* Visualisierungen zeigen nichtlineare Muster besser

>* Liniendiagramme zeigen zeitliche Muster und Ausreißer
>* Zusammenhänge kritisch prüfen, nicht kausal deuten



In [ ]:
#@title Python-Code - Zusammenhänge und Trends

# Dieses Beispiel zeigt Zusammenhänge in Streudiagrammen.
# Wir vergleichen lineare und gekrümmte Muster.
# Die Grafik macht Trends und Streuung sichtbar.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen kleine, nachvollziehbare Beispieldaten.
rng = np.random.default_rng(42)
study_hours = np.linspace(1, 10, 60)
noise = rng.normal(0, 5, size=study_hours.size)

# Der Effekt steigt zuerst stark, später langsamer.
exam_score = 35 + 18 * (1 - np.exp(-study_hours / 3)) + noise
exam_score = np.clip(exam_score, 0, 100)

# Eine Tabelle hilft beim kurzen numerischen Überblick.
data = pd.DataFrame(
    {"Lernstunden": study_hours, "Prüfungspunkte": exam_score}
)

# Wir prüfen, ob beide Spalten gleich lang sind.
if len(data["Lernstunden"]) != len(data["Prüfungspunkte"]):
    raise ValueError("Die beiden Spalten müssen gleich viele Werte enthalten.")

# Korrelation beschreibt nur lineare Tendenzen.
correlation = data["Lernstunden"].corr(data["Prüfungspunkte"])
print(f"Beobachtungen: {len(data)}")
print(f"Lineare Korrelation: {correlation:.2f}")
print("Hinweis: Die Kurve zeigt Sättigung, nicht nur eine Gerade.")

# Ein Streudiagramm zeigt Form, Richtung und Streuung gemeinsam.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(data["Lernstunden"], data["Prüfungspunkte"], alpha=0.75)

# Eine geglättete Linie macht den Trend leichter erkennbar.
trend = data.sort_values("Lernstunden")["Prüfungspunkte"].rolling(7, center=True).mean()
ax.plot(data["Lernstunden"], trend, color="orange", label="geglätteter Trend")

# Beschriftungen machen die Interpretation eindeutig.
ax.set_title("Zusammenhang zwischen Lernzeit und Prüfungsergebnis")
ax.set_xlabel("Lernstunden pro Woche")
ax.set_ylabel("Prüfungspunkte")
ax.legend()
plt.show()



### **2.3. Gruppen gezielt vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_A/image_02_03.jpg?v=1787630925" width="250">



>* Gruppen zeigen Unterschiede hinter Gesamtwerten
>* Diagrammtyp passend zur Vergleichsfrage wählen

>* Diagramm passend zu Frage und Daten wählen
>* Gruppengrößen beachten, Kategorien sinnvoll sortieren

>* Unterschiede zeigen nicht automatisch Ursachen.
>* Zusatzinformationen verhindern vorschnelle Schlüsse.



In [ ]:
#@title Python-Code - Gruppen gezielt vergleichen

# Dieses Beispiel vergleicht Gruppen mit einem Boxplot.
# Es zeigt Mittelwerte, Streuung und Ausreißer.
# Sichtbare Unterschiede werden vorsichtig interpretiert.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Wir erzeugen kleine, nachvollziehbare Beispieldaten.
rng = np.random.default_rng(42)
regions = ["Nord", "Süd", "West"]

# Jede Region bekommt eine eigene Umsatzverteilung.
north = rng.normal(loc=52, scale=6, size=35)
south = rng.normal(loc=58, scale=10, size=35)
west = rng.normal(loc=50, scale=4, size=35)

# Ein einzelner hoher Wert macht einen Ausreißer sichtbar.
south[0] = 92
sales = np.concatenate([north, south, west])

# Die Gruppennamen werden passend zu den Werten wiederholt.
region_labels = np.repeat(regions, 35)
data = pd.DataFrame({"region": region_labels, "sales": sales})

# Eine kurze Prüfung schützt vor unerwarteten Datenformen.
if data.shape != (105, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Aggregierte Kennzahlen ergänzen die grafische Verteilung.
summary = data.groupby("region")["sales"].agg(["mean", "median", "std"])
summary = summary.round(1)

# Die Tabelle bleibt klein und unterstützt den Gruppenvergleich.
print("Kennzahlen je Region, Umsatz in Tsd. Euro:")
print(summary.to_string())
print("Hinweis: Ein Unterschied im Diagramm beweist noch keine Ursache.")

# Der Boxplot zeigt Lage, Streuung und mögliche Ausreißer.
plt.figure(figsize=(7, 4))
ax = sns.boxplot(data=data, x="region", y="sales", order=regions)

# Achsentitel machen die Bedeutung der Werte klar.
ax.set_title("Umsatzverteilungen gezielt nach Region vergleichen")
ax.set_xlabel("Region")
ax.set_ylabel("Umsatz in Tsd. Euro")

# Ein dezentes Raster erleichtert das Ablesen.
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()



## **3. Beziehungen prüfen**

### **3.1. Kategorien gemeinsam betrachten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_A/image_03_01.jpg?v=1787630910" width="250">



>* Gemeinsame Kategorien zeigen verborgene Muster.
>* Auffälligkeiten können auch Datenprobleme anzeigen.

>* Absolute Zahlen immer mit Anteilen vergleichen
>* Kleine Gruppen und Zufall vorsichtig deuten

>* Zusammenhänge nicht vorschnell als Ursachen deuten
>* Fehlende Werte transparent prüfen und erklären



In [ ]:
#@title Python-Code - Kategorien gemeinsam betrachten

# Wir vergleichen zwei kategoriale Merkmale gemeinsam.
# Kreuztabellen zeigen Fallzahlen und Anteile.
# Kleine Gruppen und fehlende Werte werden sichtbar.

import pandas as pd
import matplotlib.pyplot as plt

# Diese kleinen Beispieldaten sind vollständig im Notebook enthalten.
data = pd.DataFrame(
    {
        "region": ["Nord", "Nord", "Nord", "Nord", "Süd", "Süd", "Süd", "Süd", "Süd", "West", "West", "West", "West", "West", "West", "Unbekannt", "Unbekannt", "Ost", "Ost", "Ost"],
        "contract": ["Monatlich", "Monatlich", "Jährlich", "Monatlich", "Jährlich", "Jährlich", "Monatlich", "Jährlich", "Monatlich", "Monatlich", "Jährlich", "Jährlich", "Jährlich", "Monatlich", "Jährlich", "Monatlich", "Jährlich", "Monatlich", "Monatlich", "Jährlich"],
        "churned": ["Ja", "Nein", "Nein", "Ja", "Nein", "Nein", "Nein", "Nein", "Ja", "Ja", "Nein", "Nein", "Nein", "Nein", "Nein", "Ja", "Nein", "Ja", "Ja", "Nein"],
    }
)

# Wir prüfen eine einfache Annahme zur Datenmenge.
if len(data) == 0:
    raise ValueError("Die Beispieldaten dürfen nicht leer sein.")

# Absolute Zahlen zeigen, wie viele Fälle jede Kombination enthält.
counts = pd.crosstab(data["region"], data["churned"])
counts = counts.reindex(sorted(counts.index))

# Zeilenanteile zeigen das Risiko innerhalb jeder Region.
rates = pd.crosstab(data["region"], data["churned"], normalize="index")
rates = rates.reindex(counts.index)

# Kleine Gruppen können auffällige Prozentwerte unsicher machen.
region_sizes = counts.sum(axis=1)
churn_rates = rates["Ja"].mul(100).round(1)

print("Kreuztabelle: Region gegen Kündigung")
print(counts.to_string())
print("Kündigungsanteile je Region in Prozent:")
print(churn_rates.to_string())
print("Merke: Hohe Anteile brauchen immer passende Fallzahlen.")

# Das Diagramm verbindet Anteil und Fallzahl sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(churn_rates.index, churn_rates.values, color="#4C78A8")
ax.set_title("Kündigungsanteil nach Region")
ax.set_xlabel("Region")
ax.set_ylabel("Kündigungsanteil in Prozent")

# Die Beschriftung zeigt die Bezugsgröße jeder Kategorie.
for bar, size in zip(bars, region_sizes.values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 1, f"n={size}", ha="center")

ax.set_ylim(0, max(churn_rates.values) + 15)
plt.tight_layout()
plt.show()



### **3.2. Korrelationen richtig deuten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_A/image_03_02.jpg?v=1787630912" width="250">



>* Korrelationen zeigen erste Muster zwischen Variablen.
>* Sie beweisen keine Ursache-Wirkung-Beziehung.

>* Korrelation bedeutet nicht automatisch Ursache
>* Störfaktoren und Gegenrichtungen kritisch prüfen

>* Datenqualität und Muster kritisch prüfen
>* Korrelationen vorsichtig und ohne Kausalität formulieren



In [ ]:
#@title Python-Code - Korrelationen richtig deuten

# Dieses Beispiel zeigt Korrelation ohne sichere Ursache.
# Eine Drittvariable erzeugt einen scheinbaren Zusammenhang.
# Die Grafik macht die vorsichtige Deutung sichtbar.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen kleine, nachvollziehbare Beispieldaten.
rng = np.random.default_rng(42)
temperature = np.linspace(18, 34, 40)

# Beide Größen steigen wegen der Temperatur.
ice_sales = 20 + 3.2 * temperature + rng.normal(0, 4, temperature.size)
swimming_accidents = 1 + 0.35 * temperature + rng.normal(0, 1.2, temperature.size)

# Die Tabelle bündelt die drei Variablen.
data = pd.DataFrame(
    {"Temperatur": temperature, "Eisverkäufe": ice_sales,
     "Badeunfälle": swimming_accidents}
)

# Diese Prüfung schützt vor unerwarteten Datenproblemen.
if data.shape != (40, 3):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Korrelationen beschreiben gemeinsame Bewegung, keine Ursache.
corr_sales_accidents = data["Eisverkäufe"].corr(data["Badeunfälle"])
corr_temp_sales = data["Temperatur"].corr(data["Eisverkäufe"])
corr_temp_accidents = data["Temperatur"].corr(data["Badeunfälle"])

print("Korrelation Eisverkäufe und Badeunfälle:", round(corr_sales_accidents, 2))
print("Korrelation Temperatur und Eisverkäufe:", round(corr_temp_sales, 2))
print("Korrelation Temperatur und Badeunfälle:", round(corr_temp_accidents, 2))
print("Merksatz: Korrelation ist ein Hinweis, aber kein Kausalbeweis.")

# Die Farbe zeigt die mögliche Drittvariable Temperatur.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    data["Eisverkäufe"], data["Badeunfälle"], c=data["Temperatur"], cmap="viridis"
)

ax.set_title("Scheinbarer Zusammenhang durch Temperatur")
ax.set_xlabel("Eisverkäufe pro Tag")
ax.set_ylabel("Badeunfälle pro Tag")
fig.colorbar(scatter, ax=ax, label="Temperatur in °C")
plt.show()



### **3.3. Ergebnisse sicher dokumentieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_A/image_03_03.jpg?v=1787630914" width="250">



>* Beobachtete Zusammenhänge sind keine Beweise
>* Datenbasis, Ausreißer und Grenzen dokumentieren

>* Unsicherheiten und Alternativerklärungen klar benennen
>* Korrelationen vorsichtig als Hypothesen dokumentieren

>* Ergebnisse nachvollziehbar und methodisch transparent festhalten
>* Datenrisiken vorsichtig für Entscheidungen einordnen



In [ ]:
#@title Python-Code - Ergebnisse sicher dokumentieren

# Diese Mini-EDA dokumentiert Beziehungen vorsichtig.
# Ausreißer und fehlende Werte werden sichtbar.
# Die Ausgabe trennt Beobachtung von Ursache.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen kleine, nachvollziehbare Beispieldaten.
rng = np.random.default_rng(42)
study_hours = rng.normal(5.0, 1.4, 28).round(1)
exam_score = (55 + study_hours * 6 + rng.normal(0, 6, 28)).round(1)

# Zwei Fälle zeigen typische EDA-Risiken.
study_hours = np.append(study_hours, [11.5, np.nan])
exam_score = np.append(exam_score, [62.0, 88.0])

# Die Daten stehen in einer einfachen Tabelle.
data = pd.DataFrame(
    {"study_hours": study_hours, "exam_score": exam_score}
)

# Fehlende Werte werden gezählt, nicht still ignoriert.
missing_hours = int(data["study_hours"].isna().sum())
complete_data = data.dropna(subset=["study_hours", "exam_score"])

# Ein einfacher Ausreißerhinweis nutzt den Interquartilsabstand.
q1 = complete_data["study_hours"].quantile(0.25)
q3 = complete_data["study_hours"].quantile(0.75)
iqr = q3 - q1

# Die Grenze markiert ungewöhnlich hohe Lernzeiten.
outlier_limit = q3 + 1.5 * iqr
outlier_count = int((complete_data["study_hours"] > outlier_limit).sum())

# Korrelationen werden mit und ohne Ausreißer verglichen.
correlation_all = complete_data["study_hours"].corr(complete_data["exam_score"])
without_outliers = complete_data[complete_data["study_hours"] <= outlier_limit]
correlation_clean = without_outliers["study_hours"].corr(without_outliers["exam_score"])

# Die Dokumentation nennt Datenbasis und Einschränkungen.
print(f"Datenbasis: {len(data)} Zeilen, davon {len(complete_data)} vollständig.")
print(f"Fehlende Lernzeit-Werte: {missing_hours}.")
print(f"Markierte Lernzeit-Ausreißer: {outlier_count}.")
print(f"Korrelation mit Ausreißer: {correlation_all:.2f}.")
print(f"Korrelation ohne Ausreißer: {correlation_clean:.2f}.")
print("Vorsichtige Aussage: Zusammenhang sichtbar, keine Kausalität bewiesen.")

# Die Grafik zeigt Muster, Ausreißer und fehlende Fälle.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(complete_data["study_hours"], complete_data["exam_score"], label="vollständig")

# Ausreißer werden farblich hervorgehoben.
outliers = complete_data[complete_data["study_hours"] > outlier_limit]
ax.scatter(outliers["study_hours"], outliers["exam_score"], color="red", label="Ausreißer")

# Achsen und Titel unterstützen eine sichere Interpretation.
ax.set_title("Beziehung dokumentieren: Muster, Ausreißer, Grenzen")
ax.set_xlabel("Lernzeit pro Woche in Stunden")
ax.set_ylabel("Prüfungsergebnis in Punkten")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Daten erkunden**</font>


In this lecture, you learned to:
- Erstellen aussagekräftige Tabellenübersichten und Verteilungskennzahlen. 
- Visualisieren Verteilungen, Zusammenhänge und Gruppenunterschiede. 
- Erkennen Ausreißer, fehlende Werte und Korrelation-ohne-Kausalität als EDA-Risiken. 

In the next Lecture (Lecture B), we will go over 'Manuell vorverarbeiten'